In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f <= c)

In [3]:
def cutting_plane(R,r,c,p,m,r_f,sets):
    nonstop = True
    iterations = 1
    while nonstop == True:
        [a,obj] = solvenominal(sets,p,R,r,m,r_f,c)
        print(obj)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makesetflex(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [4]:
np.random.seed(5)

In [6]:
N=5
p = (np.zeros(N)+1)*1/N
I = 1
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.04757381]
[[ 0.36649622]
 [-0.13184648]
 [-0.06832733]
 [ 0.08752065]
 [-0.01597399]]


In [7]:
r = 0
m = 0.05    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 0.1
sets = ranktoset(np.arange(N))
print(cutting_plane(R,r,c,p,m,r_f,sets))


0.04757381289491949
(array([1.]), 0.04757381289491949, 1)


In [8]:
def norisksolve(p,R,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [0<=a, a<=1, cp.sum(a)<=1]
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    

In [9]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
solvenominal (psets,p,R,r,m,r_f,c)

(array([1.]), 0.047573812896894156)

In [10]:
sets = ranktoset(np.arange(N))
sets.append([2,3])
sets.append([2,3,4])
sets.append([3,1,4])
solvenominal (sets,p,R,r,m,r_f,c)

(array([1.]), 0.04757381285193417)

In [99]:
norisksolve(p,R,r_f)

(array([1.]), 0.04757381289330838)

In [93]:
def riskcalc(a,R,p,alfa,r_f):
    x = R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.max(x) > 0:
        extra = np.max(x)
        x = x - np.max(x)
    x = -x
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk - extra
    print("the nominal risk of a:", risk)

In [94]:
a = norisksolve(p,R,r_f)[0]
riskcalc(a,R,p,m,r_f)

the nominal risk of a: -0.03078842286599226


In [14]:
p.dot(R)*a

array([0.04757381])

array([[ 0.36649622],
       [-0.13184648],
       [-0.06832733],
       [ 0.08752065],
       [-0.01597399]])

In [101]:
z = max(max(R),0)
Y = -(R - z)
m = 0.05
V=h_3(p[1],m)*Y[1]+(h_3(p[1]+p[2],m)-h_3(p[1],m))*Y[2]+(h_3(p[1]+p[2]+p[4],m)-h_3(p[1]+p[2],m))*Y[4]\
+(h_3(p[1]+p[2]+p[4]+p[3],m)-h_3(p[1]+p[2]+p[4],m))*Y[3]+(h_3(p[1]+p[2]+p[4]+p[3]+p[0],m)-h_3(p[1]+p[2]+p[4]+p[3],m))*Y[0]
V-z

array([-0.03078842])

In [22]:
np.argsort(R.dot(a))

array([1, 2, 4, 3, 0], dtype=int64)

In [105]:
h_3(1/3,0.05)*10.3+(h_3(2/3,0.05)-h_3(1/3,0.05))*10.2

7.192982456140351

In [106]:
h_3(2/3,0.05)-h_3(1/3,0.05)

0.3508771929824561